In [38]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Evaluation of TLS instrumentation approaches using PII observed

Summary statistics for PII observations across instrumentation approaches:
unique PII types, total PII transmissions, data recipients, and apps.

**Paper reference:** $\S$ 6.2, Figure 2

In [1]:
# --- Read the dataset ----
import pandas as pd
from tabulate import tabulate

from analysis_utils.shared_setup import *

orig_pii_df = pd.read_parquet("../data/pii_observed.parquet")
print(f"PII DF cols: {orig_pii_df.columns} | {orig_pii_df.size}")
pii_df = orig_pii_df.drop(columns=[
    "ip", "port", "decoders", "pii_label",
    "http_stream_type", "app_test", "clientrandom",
])

inst_approaches = sorted(pii_df["vantage_type"].unique())


PII DF cols: Index(['app', 'host', 'ip', 'port', 'pii_type', 'time', 'dir', 'uuid',
       'decoders', 'pii_label', 'http_stream_type', 'vantage_type', 'app_test',
       'clientrandom', 'tls_lib', 'company'],
      dtype='str') | 1235872


## Dataset Description

We first show how much better is Advaita per PII Type? Another way of saying 
this, is how many unique PII type transmissions were reported only by 
Advaita?

We measure this to show how much of transmissions were not reported by the
techniques per PII Type. A existing tool might report a DR receiving Ad ID
but not App Set ID. However, if both were transmitted, we should report both
since otherwise it may be a violation of Google's privacy policy.    

In [40]:
# --- Cell: PII detection overview ---
overview_df = (
    pii_df
    .groupby("vantage_type")
    .agg(
        pii_types=("pii_type", "nunique"),
        transmissions=("pii_type", "count"),
        data_recipients=("company", "nunique"),
        requests=("uuid", "nunique"),
    )
    .reindex(inst_approaches)
    .reset_index()
    .rename(columns={"vantage_type": "Approach"})
)

mark_print("**Per-approach PII detection summary:**")
mark_print(tabulate(overview_df, headers="keys", tablefmt="pipe", showindex=False))

# Advaita-exclusive DRs
dr_sets = {
    vp: set(pii_df[pii_df["vantage_type"] == vp]["company"].unique())
    for vp in inst_approaches
}
all_drs = set().union(*dr_sets.values())
existing_union = set().union(*(dr_sets[vp] for vp in inst_approaches if vp != "Advaita"))
adv_exclusive = dr_sets["Advaita"] - existing_union
best_existing = max(len(dr_sets[vp]) for vp in inst_approaches if vp != "Advaita")
best_existing_name = max(
    (vp for vp in inst_approaches if vp != "Advaita"),
    key=lambda vp: len(dr_sets[vp]),
)

mark_print(
    f"Across all approaches, **{len(all_drs)}** unique data recipients transmit PII. "
    f"Advaita detects **{len(dr_sets['Advaita'])}** DRs, "
    f"**{len(adv_exclusive)}/{len(all_drs)}** exclusively "
    f"(**{len(adv_exclusive) / best_existing * 100:.1f}%** more than the best existing ({len(adv_exclusive)} / {best_existing}), "
    f"{best_existing_name} with {best_existing}): "
    f"{', '.join(sorted(adv_exclusive))}."
)

**Per-approach PII detection summary:**

| Approach   |   pii_types |   transmissions |   data_recipients |   requests |
|:-----------|------------:|----------------:|------------------:|-----------:|
| Advaita    |          16 |           31047 |               151 |      14885 |
| Aosp-Sci   |          11 |           12603 |                87 |       5573 |
| Frida      |          10 |           14998 |               121 |       7879 |
| Mitm       |          11 |           18594 |               122 |      10094 |

Across all approaches, **185** unique data recipients transmit PII. Advaita detects **151** DRs, **34/185** exclusively (**27.9%** more than the best existing (34 / 122), Mitm with 122): 42Trck, AdView, Adex Ukraine, Admixer, Akamai, Antibanads, Axonix, BCH Brands, Basis, Button, Display.io, Funsdata-JS, Jampp, Liftoff, M2Catalyst (Network testing), Magnite, MobileFuse, Ogury, Perfect Body, PostHog, Pubeasy, RTB Stack, RhythmOne, Rush Street Interactive, Sentry, Serverbid, Simpli.fi, SmartHub, Snapchat, SpeedChecker, Synchrony Financial, Tealium, XLG Media, hCaptcha.

In [41]:
from IPython.display import display, Markdown

# Total unique PII types and DRs
df = pii_df.copy()
total_pii_types = df['pii_type'].nunique()
total_drs = df['company'].nunique()

# Per-vantage stats
vantage_stats = {}
for vp in inst_approaches:
    vp_df = df[df['vantage_type'] == vp]
    vp_pii = vp_df['pii_type'].nunique()
    vp_drs = vp_df['company'].nunique()
    vantage_stats[vp] = {
        'pii_types': vp_pii,
        'pii_pct': vp_pii / total_pii_types * 100,
        'drs': vp_drs,
        'transmissions': len(vp_df),
    }

# Exclusive DRs per vantage
exclusive_stats = {}
for vp in inst_approaches:
    vp_companies = set(df[df['vantage_type'] == vp]['company'].unique())
    other_companies = set()
    for other_vp in inst_approaches:
        if other_vp != vp:
            other_companies |= set(df[df['vantage_type'] == other_vp]['company'].unique())
    exclusive = vp_companies - other_companies
    exclusive_stats[vp] = {
        'count': len(exclusive),
        'companies': sorted(exclusive),
    }


# Per PII type: transmission counts and DR counts per vantage
pii_type_stats = {}
for pii_type in df['pii_type'].unique():
    sub = df[df['pii_type'] == pii_type]
    total_tx = len(sub)
    total_dr = sub['company'].nunique()
    per_vp = {}
    for vp in inst_approaches:
        vp_sub = sub[sub['vantage_type'] == vp]
        per_vp[vp] = {
            'tx': len(vp_sub),
            'drs': vp_sub['company'].nunique(),
        }
    pii_type_stats[pii_type] = {
        'total_tx': total_tx,
        'total_drs': total_dr,
        'per_vp': per_vp,
    }

mark_print("### Overall PII Summary")
vp_sentences = []
for vp in inst_approaches:
    s = vantage_stats[vp]
    dr_pct = s['drs'] / total_drs * 100
    vp_sentences.append(
        f"{vp} detected {s['drs']} DRs ({dr_pct:.1f}\\%) "
        f"across {s['pii_types']} PII types with {s['transmissions']:,} transmissions"
    )

exclusive_sentences = []
for vp in inst_approaches:
    e = exclusive_stats[vp]
    if e['count'] > 0:
        exclusive_sentences.append(f"{e['count']} were exclusive to {vp}")

# Advaita improvement over next best
advaita_tx = vantage_stats['Advaita']['transmissions']
others_tx = {vp: vantage_stats[vp]['transmissions'] for vp in inst_approaches if vp != 'Advaita'}
best_other_vp = max(others_tx, key=others_tx.get)
best_other_tx = others_tx[best_other_vp]
advaita_improvement = (advaita_tx - best_other_tx) / best_other_tx * 100

advaita_drs = vantage_stats['Advaita']['drs']
others_drs = {vp: vantage_stats[vp]['drs'] for vp in inst_approaches if vp != 'Advaita'}
best_other_dr_vp = max(others_drs, key=others_drs.get)
best_other_drs = others_drs[best_other_dr_vp]
advaita_dr_improvement = (advaita_drs - best_other_drs) / best_other_drs * 100


mark_print(f"""
We identified a total of {total_pii_types} PII types transmitted across \
{total_drs} different data recipients. Among the vantage points, \
{', '.join(vp_sentences[:-1])}, and {vp_sentences[-1]}. Advaita captured \
{advaita_improvement:.1f}\\% more transmissions than the next best vantage \
point ({best_other_vp}) and detected {advaita_dr_improvement:.1f}\\% more \
data recipients ({advaita_drs} vs.\\ {best_other_drs}). \
Of the {total_drs} data recipients transmitting PII, \
{', '.join(exclusive_sentences[:-1])}, and {exclusive_sentences[-1]}.
""")

# Grouped on column: 'company'
total_df = pii_df.copy().groupby(['company']).count().reset_index()[['company']]
# Performed 1 aggregation grouped on column: 'vantage_type'
vantage_df = pii_df.copy().groupby(['vantage_type']).agg(company_nunique=('company', 'nunique')).reset_index()

mark_print(f"We found a total of {len(total_df)} DRs across vantages; DRs reported by each vantages")
vantage_df

### Overall PII Summary


We identified a total of 16 PII types transmitted across 185 different data recipients. Among the vantage points, Advaita detected 151 DRs (81.6\%) across 16 PII types with 31,047 transmissions, Aosp-Sci detected 87 DRs (47.0\%) across 11 PII types with 12,603 transmissions, Frida detected 121 DRs (65.4\%) across 10 PII types with 14,998 transmissions, and Mitm detected 122 DRs (65.9\%) across 11 PII types with 18,594 transmissions. Advaita captured 67.0\% more transmissions than the next best vantage point (Mitm) and detected 23.8\% more data recipients (151 vs.\ 122). Of the 185 data recipients transmitting PII, 34 were exclusive to Advaita, 4 were exclusive to Aosp-Sci, 12 were exclusive to Frida, and 6 were exclusive to Mitm.


We found a total of 185 DRs across vantages; DRs reported by each vantages

,vantage_type,company_nunique
0,Advaita,151
1,Aosp-Sci,87
2,Frida,121
3,Mitm,122


## Statistical Testing

### Cochran's Q Test: Overall Binary PII-transmitting DR detection comparison

Cochran's Q tests whether the proportion of data recipients detected
transmitting PII differs across the four instrumentation approaches.

**Paper reference:** $\S$ 6.2 (para 1)

In [42]:
import numpy as np
from analysis_utils.stats import *

cochran_pii = perform_cochran(pii_df, inst_approaches, dropna_cols=["pii_type", "company"])

mark_print(
    f"**Cochran's Q test (binary PII detection)**\n\n"
    f"$H_0$: {cochran_pii['hypothesis']}\n\n"
    f"$Q$ = {cochran_pii['Q']:.2f}, df = {cochran_pii['df']}, "
    f"*p* = {safe_log_p(cochran_pii['p'])}, "
    f"Kendall's W = {cochran_pii['W']:.4f}, "
    f"*N* = {cochran_pii['N']} data recipients.\n\n"
    f"**{cochran_pii['verdict']}:** {cochran_pii['consequence']}"
)


**Cochran's Q test (binary PII detection)**

$H_0$: The proportion of data recipients detected is equal across all instrumentation approaches.

$Q$ = 65.06, df = 3, *p* = $<10^{-13}$, Kendall's W = 0.1172, *N* = 185 data recipients.

**Reject $H_0$:** The four approaches differ significantly in which data recipients they detect. At least one approach observes a structurally different subset of DRs.

**Pairwise McNemar Tests: PII Detection**

McNemar tests with Bonferroni correction determine which pairs of
approaches differ significantly in binary PII-transmitting DR detection.
Cell values show discordant counts (A detects / B detects) with p-values.

In [43]:
# --- Cell: McNemar pairwise (PII) ---
mcnemar_pii = perform_mcnemar_pairwise(cochran_pii["binary_matrix"], inst_approaches)

# Build display matrix from results
ordered_vps = sorted(inst_approaches)
paper_matrix = pd.DataFrame("", index=ordered_vps, columns=ordered_vps)

for r in mcnemar_pii["results"]:
    a, b = r["pair"]
    sig = "$^{***}$" if r["significant"] else ""
    p_str = (safe_log_p(r["p_corrected"]) if r["p_corrected"] < 0.001
             else f"$p={r['p_corrected']:.3f}$")
    paper_matrix.loc[a, b] = f"{r['a_not_b']}/{r['b_not_a']} ({p_str}){sig}"

for v in ordered_vps:
    paper_matrix.loc[v, v] = "---"

mark_print(
    f"**Pairwise McNemar tests** "
    f"({mcnemar_pii['n_comparisons']} comparisons, Bonferroni-adjusted $p$, "
    f"$^{{***}}$ adjusted $p<0.05$)\n\n"
    # f"Bonferroni-corrected $\\alpha$ = {mcnemar_pii['corrected_alpha']:.4f})\n\n"
    f"$H_0$: {mcnemar_pii['hypothesis']}\n\n"
    f"**{mcnemar_pii['verdict']}:** {mcnemar_pii['consequence']}"
)
mark_print(tabulate(
    paper_matrix.reset_index().rename(columns={"index": ""}),
    headers="keys", tablefmt="pipe", showindex=False,
))

**Pairwise McNemar tests** (6 comparisons, Bonferroni-adjusted $p$, $^{***}$ adjusted $p<0.05$)

$H_0$: For each pair, the probability of detection by one approach equals the probability of detection by the other.

**Reject $H_0$ for 5/6 pairs:** The following pairs do not differ significantly: Frida vs. Mitm.

|          | Advaita   | Aosp-Sci                   | Frida                      | Mitm                       |
|:---------|:----------|:---------------------------|:---------------------------|:---------------------------|
| Advaita  | ---       | 71/7 ($<10^{-11}$)$^{***}$ | 54/24 ($p=0.006$)$^{***}$  | 47/18 ($p=0.003$)$^{***}$  |
| Aosp-Sci |           | ---                        | 10/44 ($<10^{-4}$)$^{***}$ | 12/47 ($<10^{-4}$)$^{***}$ |
| Frida    |           |                            | ---                        | 22/23 ($p=1.000$)          |
| Mitm     |           |                            |                            | ---                        |

Post-hoc pairwise McNemar tests with Bonferroni correction (adjusted $p$ reported throughout) find five of the six pairwise differences significant, yielding the partial ordering Advaita > {Frida, Mitm} > Aosp-Sci.

### Friedman Test: PII Transmission Counts per (DR, PII Type) Pair

**Paper reference:** $\S$ 6.2 (para 1)

Unlike the HTTP stream analysis where the unit of observation was
the data recipient alone, here we consider (DR, PII type) pairs.
This answers a different question: do approaches differ in whether
they detect a *specific* PII type at a *specific* data recipient? More specifically, we try to answer the question:

> Do vantage types differ in whether they detect a specific PII type at a specific company?

Then we use `['company', 'pii_type']` as the pivot index and run the evaluation. 

Also, the low effect size $\\W = 0.1172\\$ from Cochran's test shows that there 
is very little correlation between the different instrumentation techniques. 
This is another reason why, for the Friedman Test, we include both the PII Types and the Company as 
indices, since we are interested in both results.

In [44]:
# --- Cell: Friedman on (DR, PII type) pairs ---
from analysis_utils.stats import perform_friedman, build_summary_table

friedman_pii = perform_friedman(
    pii_df, inst_approaches, pivot_index=["company", "pii_type"]
)

mark_print(
    f"**Friedman test (PII transmissions per (DR, PII type) pair)**\n\n"
    f"$H_0$: {friedman_pii['hypothesis']}\n\n"
    f"Number of (DR, PII type) pairs: {len(friedman_pii['count_pivot'])}. "
    f"$\\chi^2_F$ = {friedman_pii['chi2']:.2f}, "
    f"*p* = {safe_log_p(friedman_pii['p'])}.\n\n"
    f"**{friedman_pii['verdict']}:** {friedman_pii['consequence']}"
)
# friedman_pii

# Nemenyi post-hoc
nemenyi = friedman_pii["nemenyi"]
nemenyi_fmt = nemenyi.astype(str).copy()
for col in nemenyi.columns:
    for idx in nemenyi.index:
        val = nemenyi.loc[idx, col]
        if idx == col:
            nemenyi_fmt.loc[idx, col] = "---"
        else:
            nemenyi_fmt.loc[idx, col] = safe_log_p(val) if val < 0.01 else f"{val:.3f}"

mark_print("**Nemenyi post-hoc pairwise comparisons:**")
mark_print(tabulate(
    nemenyi_fmt.reset_index().rename(columns={"index": ""}),
    headers="keys", tablefmt="pipe", showindex=False,
))
# Mean ranks
mean_ranks = ', '.join(f'{vp} ({friedman_pii["mean_ranks"][vp]:.2f})' for vp in friedman_pii['mean_ranks'].sort_values(ascending=False).index)
mark_print(
    f"**Mean ranks:** "
    f"{mean_ranks}.\n\n"
)

**Friedman test (PII transmissions per (DR, PII type) pair)**

$H_0$: The median count per company x pii_type does not differ across instrumentation approaches.

Number of (DR, PII type) pairs: 371. $\chi^2_F$ = 112.19, *p* = $<10^{-23}$.

**Reject $H_0$:** The median count per company x pii_type differs significantly across approaches. Approaches that detect more subjects also capture more observations per subject, indicating deeper observability rather than mere breadth.

**Nemenyi post-hoc pairwise comparisons:**

| vantage_type   | Advaita     | Aosp-Sci    | Frida       | Mitm       |
|:---------------|:------------|:------------|:------------|:-----------|
| Advaita        | ---         | $<10^{-99}$ | $<10^{-11}$ | $<10^{-8}$ |
| Aosp-Sci       | $<10^{-99}$ | ---         | 0.433       | 0.066      |
| Frida          | $<10^{-11}$ | 0.433       | ---         | 0.776      |
| Mitm           | $<10^{-8}$  | 0.066       | 0.776       | ---        |

**Mean ranks:** Advaita (3.03), Mitm (2.43), Frida (2.34), Aosp-Sci (2.20).



In [45]:

summary = build_summary_table(cochran_pii["binary_matrix"], friedman_pii["count_pivot"], inst_approaches)
# Combined summary table (Table 4)
sorted_vps = summary["table"].index.tolist()

binary_mean_ranks = ', '.join(f'{v} ({summary["binary_mean_ranks"][v]:.2f})' for v in sorted_vps)
count_mean_ranks = ', '.join(f'{v} ({summary["count_mean_ranks"][v]:.2f})' for v in sorted_vps)

lines = "Both tests yield the same strict ordering: "
lines += f"{' > '.join(sorted_vps)}, "
lines += "with mean detection ranks of "
lines += f"{binary_mean_ranks}"
lines += " and mean stream count ranks of "
lines += f"{count_mean_ranks}."
mark_print(lines)

mark_print("**Table 3: Overall detection comparison.**")
mark_print("**Paper reference:** $\\S6.2$, para 1")
summary["table"].reset_index().rename(columns={"index": "Approach"})

Both tests yield the same strict ordering: Advaita > Mitm > Frida > Aosp-Sci, with mean detection ranks of Advaita (2.83), Mitm (2.52), Frida (2.51), Aosp-Sci (2.14) and mean stream count ranks of Advaita (3.03), Mitm (2.43), Frida (2.34), Aosp-Sci (2.20).

**Table 3: Overall detection comparison.**

**Paper reference:** $\S6.2$, para 1

,Approach,DRs Detected,Coverage (%),Mean Rank (Q),Mean Rank (F)
0,Advaita,151/185,81.6,2.83,3.03
1,Mitm,122/185,65.9,2.52,2.43
2,Frida,121/185,65.4,2.51,2.34
3,Aosp-Sci,87/185,47.0,2.14,2.20


## Within category and Between category analysis

We use the following mapping for PII types found. We stratify PII Types into 4 categories: Device, Installation, Account, and Location.

In [46]:
# PII type grouping for between-category notebooks
pii_group_map = {
    'Advertising ID': 'Device',
    'Android ID': 'Device',
    'GSF ID': 'Device',
    'Device Serial': 'Device',
    'Device IMEI': 'Device',
    'App Set ID': 'Installation',
    'Firebase Installation ID': 'Installation',
    'Firebase App Instance ID': 'Installation',
    'Google Account Name': 'Account',
    'Google Account Email': 'Account',
    'Gmail Profile Picture': 'Account',
    'Contact Phone': 'Account',
    'Contact Email': 'Account',
    'Contact Address': 'Account',
    'Photos Copyright': 'Account',
    'Location': 'Location',
    'Router Location': 'Location',
    'Video Location': 'Location',
    'Photos Location': 'Location',
}

mapping = []
for key, value in pii_group_map.items():
    mapping.append({'Category': value, 'PII Type': key})
pii_cat_df = pd.DataFrame(mapping).set_index('PII Type')

pii_cat_df

,Category
PII Type,
Advertising ID,Device
Android ID,Device
GSF ID,Device
Device Serial,Device
Device IMEI,Device
App Set ID,Installation
Firebase Installation ID,Installation
Firebase App Instance ID,Installation
Google Account Name,Account


In [47]:
# --- Cell: PII group detection matrix ---

pii_stats_df = pii_df.copy()
all_pairs = pii_stats_df[["company", "pii_type"]].drop_duplicates().copy()

for vp in inst_approaches:
    vp_subset = pii_stats_df[pii_stats_df["vantage_type"] == vp]
    vp_pairs = set(zip(vp_subset["company"], vp_subset["pii_type"]))
    all_pairs[f"{vp}_detected"] = [
        1 if p in vp_pairs else 0
        for p in zip(all_pairs["company"], all_pairs["pii_type"])
    ]

all_pairs["pii_group"] = all_pairs["pii_type"].map(pii_group_map)

# Validate mapping
unmapped = all_pairs[all_pairs["pii_group"].isna()]["pii_type"].unique()
if len(unmapped) > 0:
    mark_print(f"**WARNING:** Unmapped PII types: {', '.join(unmapped)}")

all_pairs_grouped = all_pairs.dropna(subset=["pii_group"])
pii_groups = sorted(all_pairs_grouped["pii_group"].unique())

# Summary
group_counts = (
    all_pairs_grouped
    .groupby("pii_group")
    .agg(
        pii_types=("pii_type", "nunique"),
        pairs=("company", "count"),
    )
    .sort_index()
)

mark_print(
    f"**{len(pii_groups)}** PII groups covering "
    f"**{all_pairs_grouped['pii_type'].nunique()}** PII types "
    f"and **{len(all_pairs_grouped)}** (DR, PII type) pairs."
)

group_counts

**4** PII groups covering **16** PII types and **371** (DR, PII type) pairs.

,pii_types,pairs
pii_group,,
Account,5,67
Device,4,203
Installation,3,74
Location,4,27


### Between Category Analysis

We use the chi squared test for perform between PII categories (grouped) per instrumentation technique


In [48]:
# --- Cell: Between-category chi-squared (PII groups) ---
pii_grouped_df = pii_df.copy()
pii_grouped_df["pii_group"] = pii_grouped_df["pii_type"].map(pii_group_map)
pii_grouped_df = pii_grouped_df.dropna(subset=["pii_group"])

between_pii = perform_between_category(
    pii_grouped_df, inst_approaches,
    stratify_col="pii_group",
    subject_cols=["company", "pii_type"],
)

adv_stats = None

for vp in inst_approaches:
    b = between_pii[vp]
    if b is None:
        mark_print(f"**{vp}:** Degenerate contingency table (N/A).")
        continue
    if vp == "Advaita":
        adv_stats = b
    p_vals = safe_log_p(b['p']) if b['p'] < 0.001 else f'{b["p"]:.3f}'

    mark_print(
        f"**{vp}:** $H_0$: {b['hypothesis']} "
        f"$\\chi^2$ = {b['chi2']:.2f}, df = {b['dof']}, "
        f"*p* = {p_vals}. "
        f"Min expected count: {b['min_expected']:.1f}. "
        f"**{b['verdict']}:** {b['consequence']}"
    )

**Advaita:** $H_0$: Advaita's detection rate is independent of pii_group. $\chi^2$ = 2.63, df = 3, *p* = 0.452. Min expected count: 4.5. **Fail to reject $H_0$:** No significant dependence of Advaita's detection on pii_group at alpha = 0.05. This approach provides uniform coverage across categories.

**Aosp-Sci:** $H_0$: Aosp-Sci's detection rate is independent of pii_group. $\chi^2$ = 43.17, df = 3, *p* = $<10^{-8}$. Min expected count: 11.6. **Reject $H_0$:** Aosp-Sci's detection depends on pii_group.

**Frida:** $H_0$: Frida's detection rate is independent of pii_group. $\chi^2$ = 64.77, df = 3, *p* = $<10^{-13}$. Min expected count: 11.7. **Reject $H_0$:** Frida's detection depends on pii_group.

**Mitm:** $H_0$: Mitm's detection rate is independent of pii_group. $\chi^2$ = 51.70, df = 3, *p* = $<10^{-10}$. Min expected count: 10.7. **Reject $H_0$:** Mitm's detection depends on pii_group.

### Within-Category Cochran's Q: PII Groups

For each PII group, Cochran's Q tests whether detection rates differ
across the four instrumentation approaches. This reveals whether the
observability gap is uniform across PII categories or concentrated
in specific types.

**Paper reference:** $\S$ 6.2

In [49]:
# --- Cell: Within-category Cochran's Q (PII groups) ---
pii_grouped_df = pii_df.copy()
pii_grouped_df["pii_group"] = pii_grouped_df["pii_type"].map(pii_group_map)
pii_grouped_df = pii_grouped_df.dropna(subset=["pii_group"])

within_pii = perform_within_category(
    pii_grouped_df, inst_approaches,
    stratify_col="pii_group",
    subject_cols=["company", "pii_type"],
)

# Summary table
table_rows = []
for w in within_pii:
    rates = {
        vp: w["binary_matrix"][vp].sum() / w["N"] * 100
        for vp in inst_approaches
    }
    row = {"PII Group": f"{w['category']} (n={w['N']})"}
    for vp in inst_approaches:
        row[vp] = f"{rates[vp]:.1f}%"
    row["Q"] = f"{w['Q']:.1f}" if not np.isnan(w["Q"]) else "---"
    row["p"] = safe_log_p(w["p"]) if w["p"] < 0.001 else f"{w['p']:.3f}"
    table_rows.append(row)

mark_print("**Table 6: Within-category detection rates by PII group:**")
mark_print(tabulate(table_rows, headers="keys", tablefmt="pipe", showindex=False))

# Per-group verdicts
for w in within_pii:
    if np.isnan(w["Q"]):
        mark_print(
            f"**{w['category']}** (n={w['N']}): "
            f"Q not computable (no discordance)."
        )
        continue
    mark_print(
        f"**{w['category']}** (n={w['N']}): "
        f"$H_0$: {w['hypothesis']} "
        f"$Q$ = {w['Q']:.2f}, *p* = {safe_log_p(w['p'])}. "
        f"**{w['verdict']}:** {w['consequence']}"
    )

**Table 6: Within-category detection rates by PII group:**

| PII Group           | Advaita   | Aosp-Sci   | Frida   | Mitm   |    Q | p           |
|:--------------------|:----------|:-----------|:--------|:-------|-----:|:------------|
| Account (n=67)      | 88.1%     | 17.9%      | 20.9%   | 29.9%  | 80.3 | $<10^{-16}$ |
| Device (n=203)      | 80.8%     | 46.3%      | 67.0%   | 68.0%  | 70.9 | $<10^{-14}$ |
| Installation (n=74) | 83.8%     | 66.2%      | 73.0%   | 78.4%  |  8.8 | 0.033       |
| Location (n=27)     | 88.9%     | 14.8%      | 22.2%   | 29.6%  | 35   | $<10^{-6}$  |

**Account** (n=67): $H_0$: Detection rates do not differ across approaches for Account. $Q$ = 80.26, *p* = $<10^{-16}$. **Reject $H_0$:** Approaches differ significantly in Account detection. The observability gap persists within this category.

**Device** (n=203): $H_0$: Detection rates do not differ across approaches for Device. $Q$ = 70.87, *p* = $<10^{-14}$. **Reject $H_0$:** Approaches differ significantly in Device detection. The observability gap persists within this category.

**Installation** (n=74): $H_0$: Detection rates do not differ across approaches for Installation. $Q$ = 8.76, *p* = $<10^{-1}$. **Reject $H_0$:** Approaches differ significantly in Installation detection. The observability gap persists within this category.

**Location** (n=27): $H_0$: Detection rates do not differ across approaches for Location. $Q$ = 35.02, *p* = $<10^{-6}$. **Reject $H_0$:** Approaches differ significantly in Location detection. The observability gap persists within this category.

## Visualization of results



In [50]:
from analysis_utils.shared_setup import *
import pandas as pd

fig_df = pd.read_parquet("../data/pii_top_25.parquet")
mark_print("### PII Ex-filtration by selected data recipients across TLS Instrumentation Techniques (Fig. 2)")
fig_all = plot_glyph_matrix(fig_df, y_spacing=1, x_spacing=1.1)
fig_all.update_layout(
        width=1200,
        height=700,
    )
fig_all.update_layout(
        width=1100,
        height=700,
    )
fig_all.show()
# fig_all.write_image("../images/pii_dr_paper.pdf", scale=1)

### PII Ex-filtration by selected data recipients across TLS Instrumentation Techniques (Fig. 2)

height: 640


# Supplementary PII Analysis

The following table summarizes PII transmissions across all four instrumentation approaches. Advaita observes the highest volume across all PII types, recording 31,047 transmissions, compared to 18,594 (Mitm), 14,998 (Frida), and 12,603 (Aosp-Sci). 

Advaita detects 151 DRs transmitting PII, 34 of which are missed by every other technique. At the finer (DR, PII type) granularity, 80 of 371 pairs are observed only by Advaita; a single DR can account for several such pairs when it receives multiple PII types.

In [51]:

# Unique DRs per (pii_type, vantage)
dr_ct = (
    pii_df
    .groupby(["pii_type", "vantage_type"])["company"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=inst_approaches, fill_value=0)
)

# Raw transmission counts
tx_ct = (
    pii_df
    .groupby(["pii_type", "vantage_type"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=inst_approaches, fill_value=0)
)

# Advaita-exclusive DRs per PII type
advaita_only = {}
for pii_type in pii_df["pii_type"].unique():
    sub = pii_df[pii_df["pii_type"] == pii_type]
    dr_sets = {
        vp: set(sub[sub["vantage_type"] == vp]["company"])
        for vp in inst_approaches
    }
    existing_union = set().union(*(dr_sets[v] for v in inst_approaches if v != "Advaita"))
    exclusive = dr_sets.get("Advaita", set()) - existing_union
    advaita_only[pii_type] = {"count": len(exclusive), "DRs": sorted(exclusive)}

# Combined table: "transmissions (unique DRs)"
combined = tx_ct.astype(str) + " (" + dr_ct.astype(str) + ")"
combined["Advaita-only"] = combined.index.map(
    lambda x: str(advaita_only.get(x, {}).get("count", 0))
)
combined = combined.sort_index()

# Total row
total_tx = tx_ct.sum().astype(str)

# Unique DRs per vantage point across all PII types (not the sum of per-type counts)
total_dr = (
    pii_df.groupby("vantage_type")["company"].nunique()
    .reindex(inst_approaches, fill_value=0)
    .astype(str)
)
total_row = total_tx + " (" + total_dr + ")"

# Advaita-only across all PII types: DRs that Advaita sees transmitting any PII
# and no other technique sees transmitting any PII
all_dr_sets = {
    vp: set(pii_df[pii_df["vantage_type"] == vp]["company"])
    for vp in inst_approaches
}
others_union = set().union(*(all_dr_sets[v] for v in inst_approaches if v != "Advaita"))
advaita_only_total = all_dr_sets["Advaita"] - others_union
total_row["Advaita-only"] = str(len(advaita_only_total))

combined.loc["Total"] = total_row

mark_print(
    "**Table: PII transmissions per type.** "
    "Values show transmission count (unique DRs). "
    "Advaita-only counts DRs invisible to all techniques."
)
mark_print(tabulate(
    combined.reset_index().rename(columns={"pii_type": "PII Type"}),
    headers="keys", tablefmt="pipe", showindex=False,
))

**Table: PII transmissions per type.** Values show transmission count (unique DRs). Advaita-only counts DRs invisible to all techniques.

| PII Type                 | Advaita     | Aosp-Sci   | Frida       | Mitm        |   Advaita-only |
|:-------------------------|:------------|:-----------|:------------|:------------|---------------:|
| Advertising ID           | 11577 (117) | 7457 (62)  | 7914 (92)   | 11100 (94)  |             28 |
| Android ID               | 7612 (44)   | 2544 (32)  | 3340 (44)   | 3575 (42)   |              4 |
| App Set ID               | 1621 (31)   | 1755 (22)  | 1675 (25)   | 1538 (28)   |              3 |
| Contact Email            | 9 (2)       | 0 (0)      | 0 (0)       | 0 (0)       |              2 |
| Contact Phone            | 23 (8)      | 26 (2)     | 26 (2)      | 8 (2)       |              6 |
| Device IMEI              | 63 (1)      | 0 (0)      | 0 (0)       | 0 (0)       |              1 |
| Firebase App Instance ID | 608 (4)     | 50 (3)     | 322 (3)     | 384 (4)     |              0 |
| Firebase Installation ID | 346 (27)    | 596 (24)   | 459 (26)    | 414 (26)    |              2 |
| GSF ID                   | 5 (2)       | 0 (0)      | 0 (0)       | 15 (2)      |              0 |
| Gmail Profile Picture    | 27 (8)      | 4 (2)      | 2 (1)       | 15 (4)      |              3 |
| Google Account Email     | 1441 (27)   | 51 (4)     | 109 (8)     | 86 (8)      |             14 |
| Google Account Name      | 1340 (14)   | 66 (4)     | 62 (3)      | 57 (6)      |              4 |
| Location                 | 6319 (19)   | 16 (1)     | 1089 (6)    | 1402 (8)    |              9 |
| Photos Location          | 2 (1)       | 0 (0)      | 0 (0)       | 0 (0)       |              1 |
| Router Location          | 52 (3)      | 38 (3)     | 0 (0)       | 0 (0)       |              2 |
| Video Location           | 2 (1)       | 0 (0)      | 0 (0)       | 0 (0)       |              1 |
| Total                    | 31047 (151) | 12603 (87) | 14998 (121) | 18594 (122) |             34 |

In [52]:
# Summary narrative
total_exclusive = sum(v["count"] for v in advaita_only.values())
exclusive_types = sorted([t for t, v in advaita_only.items() if v["count"] > 0])

mark_print(
    f"Advaita detects **{total_exclusive}** DR-PII pairs exclusively, "
    f"spanning {len(exclusive_types)} PII types."
)

# Advaita-exclusive detail
if exclusive_types:
    detail_rows = []
    for pii_type in exclusive_types:
        info = advaita_only[pii_type]
        detail_rows.append({
            "PII Type": pii_type,
            "Exclusive DRs": info["count"],
            "Data Recipients": ", ".join(info["DRs"]),
        })
    mark_print("**Advaita-exclusive DR-PII pairs by type:**")
    mark_print(tabulate(detail_rows, headers="keys", tablefmt="pipe", showindex=False))

Advaita detects **80** DR-PII pairs exclusively, spanning 14 PII types.

**Advaita-exclusive DR-PII pairs by type:**

| PII Type                 |   Exclusive DRs | Data Recipients                                                                                                                                                                                                                                                                                                       |
|:-------------------------|----------------:|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Advertising ID           |              28 | 42Trck, AdView, Adex Ukraine, Admixer, Akamai, Antibanads, Axonix, BCH Brands, Basis, Button, Display.io, Funsdata-JS, Jampp, Liftoff, M2Catalyst (Network testing), Magnite, MobileFuse, Ogury, PostHog, Pubeasy, RTB Stack, RhythmOne, Rush Street Interactive, Serverbid, Simpli.fi, SmartHub, Snapchat, XLG Media |
| Android ID               |               4 | Adobe, MobileFuse, Snapchat, The Trade Desk                                                                                                                                                                                                                                                                           |
| App Set ID               |               3 | Funsdata-JS, Pixalate, Sentry                                                                                                                                                                                                                                                                                         |
| Contact Email            |               2 | ByteDance, Snapchat                                                                                                                                                                                                                                                                                                   |
| Contact Phone            |               6 | ByteDance, CBS/Paramount, Google, Mixpanel, Snapchat, hCaptcha                                                                                                                                                                                                                                                        |
| Device IMEI              |               1 | Handcent Next SMS                                                                                                                                                                                                                                                                                                     |
| Firebase Installation ID |               2 | Handcent Next SMS, Snapchat                                                                                                                                                                                                                                                                                           |
| Gmail Profile Picture    |               3 | ByteDance, CBS/Paramount, Unknown                                                                                                                                                                                                                                                                                     |
| Google Account Email     |              14 | Adobe, Amplitude, ByteDance, CBS/Paramount, FreeWheel, Handcent Next SMS, LiveRamp, Meta, Perfect Body, RevenueCat, Singular, Synchrony Financial, Tealium, Unknown                                                                                                                                                   |
| Google Account Name      |               4 | ByteDance, CBS/Paramount, Snapchat, Unknown                                                                                                                                                                                                                                                                           |
| Location                 |               9 | CBS/Paramount, DoubleVerify, FreeWheel, Mixpanel, SpeedChecker, Tapjoy, Tealium, Unknown, Verve                                                                                                                                                                                                                       |
| Photos Location          |               1 | Meta                                                                                                                                                                                                                                                                                                                  |
| Router Location          |               2 | Meta, SpeedChecker                                                                                                                                                                                                                                                                                                    |
| Video Location           |               1 | Meta                                                                                                                                                                                                                                                                                                                  |